In [4]:
import math
import numpy as np

from scipy.constants import elementary_charge
from RRAM import Constants

##### Función para inicializar el array de oxigenos

In [7]:
# Defino las constantes
def Init_OxigenState(espesor_dispositivo: float, atom_size: float):
    eje_x = round(espesor_dispositivo / atom_size)
    eje_y = round(espesor_dispositivo / atom_size)

    InitialOxigenState = np.zeros((eje_x, eje_y), dtype=int)

    return InitialOxigenState


matriz_inicial = Init_OxigenState(10e-9, 0.25e-9)

matriz_inicial

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

##### Función para generar los iones de oxígeno

In [28]:
def GenerateOxigen(oxigen_state: np.array, num_oxigen: int):

    eje_y = oxigen_state.shape[1]
    y = np.zeros(num_oxigen, dtype=int)

    for i in range(num_oxigen):
        # Genero las coordenadas aleatorias para el eje y donde habrá un oxígeno
        y[i] = np.random.randint(0, eje_y)

    # Itero sobre cada par coordenada para asignar el valor de 1 que representa que se generó un oxígeno en esa posición
    for i in range(num_oxigen):
        if oxigen_state[0, y[i]] == 0:
            oxigen_state[0, y[i]] = 1

    # Devuelvo la matriz con los oxígenos generados
    return oxigen_state


print(GenerateOxigen(matriz_inicial, 10))

[17  0 34  8  5 19 18 22 19 28]


##### Función para mover los oxígenos

In [29]:
# 1 Primero genero los iones de oxígeno

from math import gamma


def Move_OxigenIons(simu_time: float, oxigen_state: np.array, temperature: float, E_field: float, atom_size: float, factor: float):

    # Constants for the simulation
    E_m = Constants.E_m
    gamma_drift = Constants.gamma_drift
    k_b_ev = Constants.k_b_ev
    t_0 = Constants.t_0

    # Obtengo la velocidad de los iones de oxígeno v = (a/t0)*exp(−Em/kT) sinh(q * γ_drift * F/kT)
    senoh = math.sinh((2*elementary_charge * E_field * gamma_drift) / (k_b_ev * temperature))
    exp_velocity = math.exp(-E_m / (k_b_ev * temperature))

    # el t_0 es en verdad el valor de 1/t_0 que lo pogno directamente y el factor es algo que introduzco a mano para ajustar la velocidad
    oxigen_velocity = factor * t_0 * atom_size * (senoh * exp_velocity)

    # Calculo la cantidad de "casillas" que se moverá el ion de oxígeno
    displacement = int(round(oxigen_velocity * simu_time / atom_size))

    return oxigen_state

##### Función para recombinar iones oxígeno.